In [ ]:
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
len(documents)

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json

user_prompt = json.dumps(doc)

user_prompt

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
result = response.output_parsed

print(result)

In [ ]:
print(result.questions)

In [ ]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

In [ ]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage


In [ ]:
generate_ground_truth(doc)

In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

usages
ground_truth

  0%|          | 0/5 [00:00<?, ?it/s]

[{'question': 'I just found this course — is it too late to join, or can I still start now?',
  'document': '74eb249bbf'},
 {'question': 'Am I still allowed to enroll if I discovered the class after it already began?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the course late, and what happens if I want a certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, will I still be able to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to jump into the course this late, or do I need to have joined earlier?',
  'document': '74eb249bbf'},
 {'question': 'I signed up for the LLM Zoomcamp — when should I get the confirmation email, or is there something else I need to wait for?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need to register before I can start the course and submit homework, or can I just begin while the form is still open?',
  'document': '977bf7786c'},
 {'question': "If I filled out the signup form,